In [ ]:
# === AQR-Style Factor Screening + Mean-Variance Optimization + Monte Carlo ===
# 1. Screen stocks using Value, Momentum, Quality, Low Risk, Growth, ESG factors
# 2. Run MVO on screened stocks with multiple portfolio strategies
# 3. Monte Carlo simulation with $10k annual withdrawals starting year 4
# Run in Colab or local env. Requires: yfinance, pandas, numpy, cvxpy, scipy

import yfinance as yf
import pandas as pd
import numpy as np
import cvxpy as cp
from datetime import datetime, timedelta
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# --- 1) Tickers to Screen ---
tickers = ["QQQM","SNOW","NEE","LIT","MSFT","GLD","AMZN","NLR","META",
           "GOOG","AAPL","ASML","COST","CSCO","HOOD","LEN","PBW","RGTI",
           "SHOP","XYZ","SOFI","PLD","CRM","ICLN","PAVE","FSLR","EFRA"]
tickers = list(dict.fromkeys(tickers))

print(f"{'='*80}")
print("PHASE 1: AQR-STYLE FACTOR SCREENING")
print(f"{'='*80}")
print(f"\nScreening {len(tickers)} tickers...")

# --- 2) Download Data for Screening (5 years) ---
end = datetime.now().date()
start = end - timedelta(days=5*365 + 30)

print(f"Downloading data from {start} to {end}...")
data = yf.download(tickers, start=start.isoformat(), end=end.isoformat(),
                   progress=False, auto_adjust=True)

# Extract close prices
if len(tickers) > 1:
    if 'Close' in data.columns.get_level_values(0):
        prices = data['Close']
    else:
        prices = data
else:
    prices = data[['Close']].copy()
    prices.columns = tickers

prices = prices.dropna(axis=1, how='all').dropna(axis=0, how='any')
available_tickers = prices.columns.tolist()

print(f"Successfully loaded price data for {len(available_tickers)} tickers")

# --- 3) Calculate Factor Metrics ---
factor_data = []

for ticker in available_tickers:
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        # Get price history
        ticker_prices = prices[ticker]
        returns = ticker_prices.pct_change().dropna()

        # --- VALUE METRICS ---
        pe = info.get('trailingPE', np.nan)
        pb = info.get('priceToBook', np.nan)
        # P/FCF approximation
        fcf_ps = info.get('freeCashflow', np.nan)
        shares = info.get('sharesOutstanding', np.nan)
        price = ticker_prices.iloc[-1]
        p_fcf = (price * shares / fcf_ps) if (fcf_ps and shares and fcf_ps > 0) else np.nan
        ev_ebitda = info.get('enterpriseToEbitda', np.nan)

        # --- MOMENTUM METRICS ---
        # 6-month return (excluding last month)
        if len(ticker_prices) >= 150:
            mom_6m = (ticker_prices.iloc[-22] / ticker_prices.iloc[-150]) - 1
        else:
            mom_6m = np.nan

        # 12-month return (excluding last month)
        if len(ticker_prices) >= 275:
            mom_12m = (ticker_prices.iloc[-22] / ticker_prices.iloc[-275]) - 1
        else:
            mom_12m = np.nan

        # RSI (14-day)
        if len(returns) >= 14:
            gains = returns.where(returns > 0, 0).rolling(14).mean()
            losses = -returns.where(returns < 0, 0).rolling(14).mean()
            rs = gains / losses.replace(0, np.nan)
            rsi = 100 - (100 / (1 + rs.iloc[-1]))
        else:
            rsi = np.nan

        # % below 52-week high
        high_52w = ticker_prices.iloc[-252:].max() if len(ticker_prices) >= 252 else ticker_prices.max()
        pct_below_high = (price / high_52w) - 1

        # --- QUALITY METRICS ---
        roe = info.get('returnOnEquity', np.nan)
        roa = info.get('returnOnAssets', np.nan)
        gross_margin = info.get('grossMargins', np.nan)
        operating_margin = info.get('operatingMargins', np.nan)
        debt_to_equity = info.get('debtToEquity', np.nan)

        # Earnings stability (simplified - use available data)
        earnings_stability = np.nan  # Would need historical financials

        # --- LOW RISK METRICS ---
        # 1-year volatility
        if len(returns) >= 252:
            volatility = returns.iloc[-252:].std() * np.sqrt(252)
        else:
            volatility = returns.std() * np.sqrt(252)

        # Beta (vs SPY approximation using overall market)
        beta = info.get('beta', np.nan)

        # Max drawdown (3 years)
        if len(ticker_prices) >= 756:
            prices_3y = ticker_prices.iloc[-756:]
        else:
            prices_3y = ticker_prices
        cummax = prices_3y.cummax()
        drawdown = (prices_3y / cummax) - 1
        max_drawdown = drawdown.min()

        # Downside deviation
        negative_returns = returns[returns < 0]
        downside_dev = negative_returns.std() * np.sqrt(252) if len(negative_returns) > 0 else np.nan

        # --- GROWTH METRICS ---
        eps_growth = info.get('earningsGrowth', np.nan)
        revenue_growth = info.get('revenueGrowth', np.nan)

        # Revenue growth 3Y CAGR (simplified)
        revenue_growth_3y = info.get('revenueGrowth', np.nan)  # Approximation

        # CAPEX/Revenue
        capex = info.get('capitalExpenditures', np.nan)
        revenue = info.get('totalRevenue', np.nan)
        capex_revenue = abs(capex / revenue) if (capex and revenue and revenue > 0) else np.nan

        # --- ESG METRICS (mock data - not available in yfinance) ---
        esg_score = np.nan

        # Store data
        factor_data.append({
            'ticker': ticker,
            # Value
            'pe': pe,
            'pb': pb,
            'p_fcf': p_fcf,
            'ev_ebitda': ev_ebitda,
            # Momentum
            'mom_6m': mom_6m,
            'mom_12m': mom_12m,
            'rsi': rsi,
            'pct_below_high': pct_below_high,
            # Quality
            'roe': roe,
            'roa': roa,
            'gross_margin': gross_margin,
            'operating_margin': operating_margin,
            'debt_to_equity': debt_to_equity,
            # Low Risk
            'volatility': volatility,
            'beta': beta,
            'max_drawdown': max_drawdown,
            'downside_dev': downside_dev,
            # Growth
            'eps_growth': eps_growth,
            'revenue_growth': revenue_growth,
            'capex_revenue': capex_revenue,
            # ESG
            'esg_score': esg_score
        })

    except Exception as e:
        print(f"Error processing {ticker}: {e}")
        continue

# Convert to DataFrame
df_factors = pd.DataFrame(factor_data)
print(f"\nCalculated factors for {len(df_factors)} stocks")

# --- 4) Standardize Metrics to Z-Scores ---
def zscore_column(series):
    """Calculate z-score, handling NaN values"""
    valid = series.dropna()
    if len(valid) < 2:
        return series * 0  # Return zeros if insufficient data
    mean = valid.mean()
    std = valid.std()
    if std == 0:
        return series * 0
    return (series - mean) / std

# Value metrics (lower is better, so invert)
df_factors['z_pe'] = -zscore_column(df_factors['pe'])
df_factors['z_pb'] = -zscore_column(df_factors['pb'])
df_factors['z_p_fcf'] = -zscore_column(df_factors['p_fcf'])
df_factors['z_ev_ebitda'] = -zscore_column(df_factors['ev_ebitda'])

# Momentum (higher is better)
df_factors['z_mom_6m'] = zscore_column(df_factors['mom_6m'])
df_factors['z_mom_12m'] = zscore_column(df_factors['mom_12m'])
df_factors['z_rsi'] = zscore_column(df_factors['rsi'])
df_factors['z_pct_below_high'] = zscore_column(df_factors['pct_below_high'])

# Quality (higher is better, except debt)
df_factors['z_roe'] = zscore_column(df_factors['roe'])
df_factors['z_roa'] = zscore_column(df_factors['roa'])
df_factors['z_gross_margin'] = zscore_column(df_factors['gross_margin'])
df_factors['z_operating_margin'] = zscore_column(df_factors['operating_margin'])
df_factors['z_debt_to_equity'] = -zscore_column(df_factors['debt_to_equity'])

# Low Risk (lower is better, so invert)
df_factors['z_volatility'] = -zscore_column(df_factors['volatility'])
df_factors['z_beta'] = -zscore_column(df_factors['beta'])
df_factors['z_max_drawdown'] = -zscore_column(df_factors['max_drawdown'])
df_factors['z_downside_dev'] = -zscore_column(df_factors['downside_dev'])

# Growth (higher is better)
df_factors['z_eps_growth'] = zscore_column(df_factors['eps_growth'])
df_factors['z_revenue_growth'] = zscore_column(df_factors['revenue_growth'])
df_factors['z_capex_revenue'] = zscore_column(df_factors['capex_revenue'])

# ESG (higher is better)
df_factors['z_esg_score'] = zscore_column(df_factors['esg_score'])

# --- 5) Calculate Factor Scores ---
# Average z-scores within each factor (ignoring NaN)
df_factors['Value_Score'] = df_factors[['z_pe', 'z_pb', 'z_p_fcf', 'z_ev_ebitda']].mean(axis=1, skipna=True)
df_factors['Momentum_Score'] = df_factors[['z_mom_6m', 'z_mom_12m', 'z_rsi', 'z_pct_below_high']].mean(axis=1, skipna=True)
df_factors['Quality_Score'] = df_factors[['z_roe', 'z_roa', 'z_gross_margin', 'z_operating_margin', 'z_debt_to_equity']].mean(axis=1, skipna=True)
df_factors['LowRisk_Score'] = df_factors[['z_volatility', 'z_beta', 'z_max_drawdown', 'z_downside_dev']].mean(axis=1, skipna=True)
df_factors['Growth_Score'] = df_factors[['z_eps_growth', 'z_revenue_growth', 'z_capex_revenue']].mean(axis=1, skipna=True)
df_factors['ESG_Score'] = df_factors['z_esg_score']

# Replace NaN factor scores with 0
factor_cols = ['Value_Score', 'Momentum_Score', 'Quality_Score', 'LowRisk_Score', 'Growth_Score', 'ESG_Score']
df_factors[factor_cols] = df_factors[factor_cols].fillna(0)

# --- 6) Calculate Composite Score ---
weights = {
    'Value_Score': 0.20,
    'Momentum_Score': 0.20,
    'Quality_Score': 0.20,
    'LowRisk_Score': 0.15,
    'Growth_Score': 0.05,
    'ESG_Score': 0.20
}

df_factors['Composite_Score'] = (
    df_factors['Value_Score'] * weights['Value_Score'] +
    df_factors['Momentum_Score'] * weights['Momentum_Score'] +
    df_factors['Quality_Score'] * weights['Quality_Score'] +
    df_factors['LowRisk_Score'] * weights['LowRisk_Score'] +
    df_factors['Growth_Score'] * weights['Growth_Score'] +
    df_factors['ESG_Score'] * weights['ESG_Score']
)

# --- 7) Rank and Display Results ---
df_factors = df_factors.sort_values('Composite_Score', ascending=False)
df_factors['Rank'] = range(1, len(df_factors) + 1)

print(f"\n{'='*80}")
print("FACTOR SCREENING RESULTS - TOP 10 STOCKS")
print(f"{'='*80}")

display_cols = ['Rank', 'ticker', 'Value_Score', 'Momentum_Score', 'Quality_Score',
                'LowRisk_Score', 'Growth_Score', 'ESG_Score', 'Composite_Score']
print(df_factors[display_cols].head(10).to_string(index=False, float_format='%.3f'))

print(f"\n{'='*80}")
print("Complete Rankings:")
print(df_factors[['Rank', 'ticker', 'Composite_Score']].to_string(index=False, float_format='%.3f'))

# --- 8) Use Screened Stocks for MVO ---
print(f"\n{'='*80}")
print("PHASE 2: MEAN-VARIANCE OPTIMIZATION")
print(f"{'='*80}")

# Use all screened tickers for MVO
mvo_tickers = df_factors['ticker'].tolist()
print(f"\nUsing {len(mvo_tickers)} screened tickers for portfolio optimization")

# Refilter price data
adj = prices[mvo_tickers].dropna(axis=0, how='any')

# Calculate returns
rets = np.log(adj / adj.shift(1)).dropna()
rets = rets.replace([np.inf, -np.inf], np.nan).dropna()

trading_days = 252
mu_daily = rets.mean()
mu_annual = mu_daily * trading_days
mu_annual_simple = np.exp(mu_annual) - 1

cov_daily = rets.cov()
cov_annual = cov_daily * trading_days

# Display summary
summary = pd.DataFrame({
    'ticker': mvo_tickers,
    'factor_rank': df_factors['Rank'].values,
    'composite_score': df_factors['Composite_Score'].values,
    'annual_mean_return': mu_annual_simple.values,
    'annual_volatility': np.sqrt(np.diag(cov_annual))
})
print("\nStocks ranked by factor score with return metrics:")
print(summary.to_string(index=False, float_format='%.3f'))

# --- 9) Define Portfolio Strategies ---
target_return = 0.13

portfolio_strategies = [
    {
        'name': 'ALL TICKERS - Optimal Allocation',
        'max_weight': 0.05,
        'min_weight': 0.01,
        'diversification_penalty': 0.25,
        'force_all': True
    },
    {
        'name': 'Highly Diversified (15-20 assets)',
        'max_weight': 0.08,
        'min_weight': 0.02,
        'diversification_penalty': 0.15,
        'force_all': False
    },
    {
        'name': 'Moderately Diversified (10-15 assets)',
        'max_weight': 0.12,
        'min_weight': 0.02,
        'diversification_penalty': 0.08,
        'force_all': False
    },
    {
        'name': 'Concentrated Growth (8-12 assets)',
        'max_weight': 0.18,
        'min_weight': 0.03,
        'diversification_penalty': 0.03,
        'force_all': False
    },
    {
        'name': 'Balanced Risk (12-15 assets)',
        'max_weight': 0.10,
        'min_weight': 0.025,
        'diversification_penalty': 0.10,
        'force_all': False
    },
    {
        'name': 'Maximum Diversification (18+ assets)',
        'max_weight': 0.07,
        'min_weight': 0.015,
        'diversification_penalty': 0.20,
        'force_all': False
    }
]

n = len(mvo_tickers)
mu = mu_annual_simple.values
Sigma = cov_annual.values

# Ensure covariance matrix is symmetric
Sigma = (Sigma + Sigma.T) / 2
regularization = 1e-8
Sigma = Sigma + regularization * np.eye(n)

# Check target feasibility
max_return = mu.max()
min_return = mu.min()
print(f"\nReturn range: {min_return:.2%} to {max_return:.2%}")
print(f"Target return: {target_return:.2%}")

if target_return > max_return:
    print(f"\nWARNING: Adjusting target to {max_return * 0.95:.2%}")
    target_return = max_return * 0.95

# --- 10) Monte Carlo Simulation Function ---
def monte_carlo_simulation(weights, mu, Sigma, initial_value=500000, years=10,
                          simulations=10000, annual_withdrawal=10000, withdrawal_start_year=4):
    port_return = weights @ mu
    port_vol = np.sqrt(weights @ Sigma @ weights)

    final_values = np.zeros(simulations)
    cagrs = np.zeros(simulations)

    for sim in range(simulations):
        value = initial_value

        for year in range(1, years + 1):
            annual_return = np.random.normal(port_return, port_vol)
            value = value * (1 + annual_return)

            if year >= withdrawal_start_year:
                value = max(0, value - annual_withdrawal)

        final_values[sim] = value

        if value > 0:
            cagrs[sim] = (value / initial_value) ** (1/years) - 1
        else:
            cagrs[sim] = -1.0

    results = {
        'mean_final_value': np.mean(final_values),
        'median_final_value': np.median(final_values),
        'low_final_value': np.percentile(final_values, 5),
        'high_final_value': np.percentile(final_values, 95),
        'mean_cagr': np.mean(cagrs),
        'median_cagr': np.median(cagrs),
        'low_cagr': np.percentile(cagrs, 5),
        'high_cagr': np.percentile(cagrs, 95),
        'prob_loss': np.sum(final_values < initial_value) / simulations,
        'prob_below_200k': np.sum(final_values < 200000) / simulations,
        'all_final_values': final_values,
        'all_cagrs': cagrs
    }

    return results

# --- 11) Optimize Portfolios ---
all_results = []

print(f"\n{'='*80}")
print("OPTIMIZING PORTFOLIOS...")
print(f"{'='*80}")

for strategy in portfolio_strategies:
    print(f"\nOptimizing: {strategy['name']}...")

    max_weight = strategy['max_weight']
    min_weight = strategy['min_weight']
    div_penalty = strategy['diversification_penalty']
    force_all = strategy.get('force_all', False)

    w = cp.Variable(n, nonneg=True)

    portfolio_variance = cp.quad_form(w, Sigma)
    diversification_penalty = div_penalty * cp.sum_squares(w)
    objective = cp.Minimize(portfolio_variance + diversification_penalty)

    constraints = [
        cp.sum(w) == 1,
        w @ mu >= target_return,
        w <= max_weight
    ]

    if force_all:
        constraints.append(w >= min_weight)

    prob = cp.Problem(objective, constraints)

    try:
        prob.solve(solver=cp.OSQP, verbose=False)
    except:
        try:
            prob.solve(solver=cp.SCS, verbose=False)
        except:
            prob.solve(verbose=False)

    if prob.status not in ["optimal", "optimal_inaccurate"]:
        print(f"Warning: Optimization status: {prob.status}")
        continue

    weights = np.array(w.value).flatten()
    if not force_all:
        weights[weights < min_weight] = 0
    weights = weights / weights.sum()

    port_return = weights @ mu
    port_vol = np.sqrt(weights @ Sigma @ weights)
    sharpe = port_return / port_vol if port_vol > 0 else np.nan

    out = pd.DataFrame({
        'ticker': mvo_tickers,
        'weight': weights
    })
    out = out[out.weight > 0].sort_values('weight', ascending=False)

    capital = 500_000
    out['capital_$'] = out['weight'] * capital

    # Add factor scores to output
    out = out.merge(df_factors[['ticker', 'Rank', 'Composite_Score']], on='ticker', how='left')

    result = {
        'strategy': strategy['name'],
        'num_assets': len(out),
        'expected_return': port_return,
        'volatility': port_vol,
        'sharpe': sharpe,
        'weights': weights,
        'allocations': out
    }
    all_results.append(result)

# --- 12) Run Monte Carlo ---
print(f"\n{'='*80}")
print("PHASE 3: MONTE CARLO SIMULATIONS")
print(f"{'='*80}")
print("Running 10,000 iterations per portfolio...")
print("Projection: 10 years with $10k annual withdrawals starting year 4")

for result in all_results:
    print(f"\nSimulating: {result['strategy']}...")

    mc_results = monte_carlo_simulation(
        weights=result['weights'],
        mu=mu,
        Sigma=Sigma,
        initial_value=500000,
        years=10,
        simulations=10000,
        annual_withdrawal=10000,
        withdrawal_start_year=4
    )

    result['mc_results'] = mc_results

# --- 13) Display Results ---
print(f"\n{'='*80}")
print("DETAILED PORTFOLIO RESULTS")
print(f"{'='*80}")

for result in all_results:
    mc = result['mc_results']

    print(f"\n{'='*80}")
    print(f"{result['strategy'].upper()}")
    print(f"{'='*80}")
    print(f"Number of assets:        {result['num_assets']}")
    print(f"Expected return:         {result['expected_return']:.2%}")
    print(f"Annual volatility:       {result['volatility']:.2%}")
    print(f"Sharpe ratio (Rf=0%):    {result['sharpe']:.3f}")

    print(f"\nMONTE CARLO RESULTS (10-year projection with withdrawals):")
    print(f"  Initial investment:    $500,000")
    print(f"  Total withdrawals:     $70,000 (years 4-10)")
    print(f"\n  Final Portfolio Value:")
    print(f"    Mean:                ${mc['mean_final_value']:,.0f}")
    print(f"    Median:              ${mc['median_final_value']:,.0f}")
    print(f"    Low (5th percentile): ${mc['low_final_value']:,.0f}")
    print(f"    High (95th percentile): ${mc['high_final_value']:,.0f}")

    print(f"\n  CAGR (After Withdrawals):")
    print(f"    Mean:                {mc['mean_cagr']:.2%}")
    print(f"    Median:              {mc['median_cagr']:.2%}")
    print(f"    Low (5th percentile): {mc['low_cagr']:.2%}")
    print(f"    High (95th percentile): {mc['high_cagr']:.2%}")

    print(f"\n  Risk Metrics:")
    print(f"    Probability of loss: {mc['prob_loss']:.1%}")
    print(f"    Prob. below $200k:   {mc['prob_below_200k']:.1%}")

    print(f"\nTop Holdings (with Factor Scores):")
    display = result['allocations'][['ticker', 'weight', 'capital_$', 'Rank', 'Composite_Score']].head(10)
    print(display.to_string(index=False))

# --- 14) Final Comparison ---
print(f"\n{'='*80}")
print("MONTE CARLO COMPARISON SUMMARY")
print(f"{'='*80}")

comparison = pd.DataFrame([
    {
        'Strategy': r['strategy'],
        'Assets': r['num_assets'],
        'Sharpe': f"{r['sharpe']:.3f}",
        'Mean CAGR': f"{r['mc_results']['mean_cagr']:.2%}",
        'Median CAGR': f"{r['mc_results']['median_cagr']:.2%}",
        'Low CAGR': f"{r['mc_results']['low_cagr']:.2%}",
        'High CAGR': f"{r['mc_results']['high_cagr']:.2%}",
        'Mean Final': f"${r['mc_results']['mean_final_value']/1e3:.0f}k",
        'Median Final': f"${r['mc_results']['median_final_value']/1e3:.0f}k"
    }
    for r in all_results
])

print("\n" + comparison.to_string(index=False))

# --- 15) Recommendations ---
print(f"\n{'='*80}")
print("RECOMMENDATIONS")
print(f"{'='*80}")

best_sharpe_idx = np.argmax([r['sharpe'] for r in all_results])
best_sharpe = all_results[best_sharpe_idx]

best_cagr_idx = np.argmax([r['mc_results']['median_cagr'] for r in all_results])
best_cagr = all_results[best_cagr_idx]

best_risk_idx = np.argmin([r['mc_results']['prob_loss'] for r in all_results])
best_risk = all_results[best_risk_idx]

print(f"\n1. BEST RISK-ADJUSTED (Highest Sharpe): {best_sharpe['strategy']}")
print(f"   Sharpe: {best_sharpe['sharpe']:.3f} | Median CAGR: {best_sharpe['mc_results']['median_cagr']:.2%}")

print(f"\n2. HIGHEST RETURNS (Best Median CAGR): {best_cagr['strategy']}")
print(f"   Median CAGR: {best_cagr['mc_results']['median_cagr']:.2%} | Median Final: ${best_cagr['mc_results']['median_final_value']:,.0f}")

print(f"\n3. LOWEST RISK (Lowest Loss Probability): {best_risk['strategy']}")
print(f"   Loss Probability: {best_risk['mc_results']['prob_loss']:.1%} | Median CAGR: {best_risk['mc_results']['median_cagr']:.2%}")

print(f"\n{'='*80}")
print("Analysis complete!")
print(f"{'='*80}")

PHASE 1: AQR-STYLE FACTOR SCREENING

Screening 27 tickers...
Successfully loaded price data for 27 tickers

Calculated factors for 27 stocks

FACTOR SCREENING RESULTS - TOP 10 STOCKS
 Rank ticker  Value_Score  Momentum_Score  Quality_Score  LowRisk_Score  Growth_Score  ESG_Score  Composite_Score
    1   ICLN        0.875           0.007          0.000          0.633         0.000      0.000            0.271
    2   GOOG        0.289           0.277          0.518          0.192        -0.130      0.000            0.239
    3    PBW        0.505          -0.005          0.000          0.727         0.000      0.000            0.209
    4   META        0.182          -0.099          0.759          0.123         0.105      0.000            0.192
    5   EFRA        0.683           0.133          0.000          0.176         0.000      0.000            0.190
    6   CSCO        0.367           0.359          0.075          0.225        -0.300      0.000            0.179
    7    GLD       